# Synthetic datasets

The six examples and four-row plot come from `vae_nmf_sklearn.ipynb` and `sklearn_result.png`. I use the same dataset types and keep the simple two-dimensional latent space. Seeds and finite training budgets make the experiment repeatable.

The model can still collapse or fail to separate circles and moons. The plot shows the actual fitted result; it is not designed to reproduce the colors in the saved screenshot.


In [ ]:
%run ./functions.ipynb
from sklearn.datasets import make_circles, make_moons, make_blobs
from sklearn.preprocessing import MinMaxScaler
from matplotlib.colors import ListedColormap
from IPython.display import display


In [ ]:
seed = 42
n_samples = 600            # original comparison: 1000
representation_epochs = 50
pretrain_epochs = 20
joint_epochs = 80
batch_size = 256
joint_kind = 'vae'        # 'ae' also runs the joint AE + MF experiment
seed_everything(seed)
rng = np.random.default_rng(seed)

noisy_circles = make_circles(n_samples=n_samples, factor=.5, noise=.05,
                             random_state=seed)
noisy_moons = make_moons(n_samples=n_samples, noise=.05, random_state=seed)
blobs = make_blobs(n_samples=n_samples, centers=2, random_state=8)
no_structure = (rng.random((n_samples, 2)), np.zeros(n_samples, dtype=int))
X0, y0 = make_blobs(n_samples=n_samples, centers=2, random_state=170)
aniso = (X0 @ np.array([[.6, -.6], [-.4, .8]]), y0)
varied = make_blobs(n_samples=n_samples, centers=2, cluster_std=[1., 2.5],
                    random_state=170)
datasets_list = [noisy_circles, noisy_moons, blobs, no_structure, aniso, varied]
datasets_name = ['noisy_circles', 'noisy_moons', 'blobs', 'no_structure', 'aniso', 'varied']

# Fit input scaling once. Never detach and refit a scaler inside the joint loss.
scaled_data = [torch.tensor(MinMaxScaler().fit_transform(X), dtype=torch.float32)
               for X, y in datasets_list]
colors = ListedColormap(['#377eb8', '#ff7f00'])


## Reconstruction-only representations

This extends the AE illustration in thesis Figure 1 with the VAE and the unstructured example. The colors here are the generating labels, used only to inspect the representation. The VAE latent coordinates are posterior means.


In [ ]:
representations = {}
fig, axes = plt.subplots(3, 6, figsize=(15, 7.5))
for col, (name, (X_original, y), X) in enumerate(zip(datasets_name, datasets_list, scaled_data)):
    axes[0, col].scatter(X_original[:, 0], X_original[:, 1], c=y, cmap=colors, s=3)
    axes[0, col].set_title(name)
    for row, kind in enumerate(['ae', 'vae'], start=1):
        seed_everything(seed)
        model = (AutoEncoder([2, 10, 2], dropout=0., final_activation=nn.Identity()) if kind == 'ae'
                 else VariationalAutoEncoder([2, 10, 2], dropout=0.)).to(device)
        pre_train(X, model, epochs=representation_epochs, batch_size=batch_size,
                  loss='mse', seed=seed)
        Z = vae_encode(X, model)
        representations[(name, kind)] = {'model': model, 'Z': Z}
        axes[row, col].scatter(Z[:, 0], Z[:, 1], c=y, cmap=colors, s=3)
for ax, label in zip(axes[:, 0], ['Dataset', 'AE latent', 'VAE latent']):
    ax.set_ylabel(label)
fig.tight_layout()
plt.show()


## Joint representation and matrix factorization

$\mu(X) \approx YC$ uses signed centers and nonnegative assignments. The MF loss stays in PyTorch so it can update the encoder. The early synthetic notebook converted latent coordinates to NumPy inside this loss, which interrupted that gradient; that is corrected here.

All six examples use two requested clusters, including `no_structure`. Its two-way partition is arbitrary, so it is omitted from the accuracy table.


In [ ]:
joint_results = {}
kmeans_results = {}
records = []
for name, (X_original, y), X in zip(datasets_name, datasets_list, scaled_data):
    seed_everything(seed)
    model = (AutoEncoder([2, 10, 2], dropout=0., final_activation=nn.Identity()) if joint_kind == 'ae'
             else VariationalAutoEncoder([2, 10, 2], dropout=0.)).to(device)
    result = train_vae_mf_model_optimized(
        X, n_clusters=2, model=model, pretrain_epochs=pretrain_epochs,
        epochs=joint_epochs, batch_size=batch_size, warm_epochs=20,
        mf_weight=.6, kappa=1e-4, gamma=1e-3, loss='mse', seed=seed,
        verbose=False)
    joint_results[name] = result
    km_pred = KMeans(n_clusters=2, n_init=10, random_state=seed).fit_predict(X_original)
    kmeans_results[name] = km_pred
    if name != 'no_structure':
        records.append({'dataset': name, 'method': 'k-means', **score_dict(y, km_pred)})
        records.append({'dataset': name, 'method': joint_kind.upper() + ' + MF',
                        **score_dict(y, result['pred'])})

display(pd.DataFrame(records))


In [ ]:
fig, axes = plt.subplots(4, 6, figsize=(15, 9))
for col, (name, (X_original, y), X) in enumerate(zip(datasets_name, datasets_list, scaled_data)):
    result = joint_results[name]
    Z, pred = result['Z'], result['pred']
    axes[0, col].scatter(X_original[:, 0], X_original[:, 1], c=y, cmap=colors, s=3)
    axes[0, col].set_title(name)
    axes[1, col].scatter(X_original[:, 0], X_original[:, 1], c=kmeans_results[name], cmap=colors, s=3)
    axes[2, col].scatter(X[:, 0], X[:, 1], c=pred, cmap=colors, s=3)
    axes[3, col].scatter(Z[:, 0], Z[:, 1], c=pred, cmap=colors, s=3)
for ax, label in zip(axes[:, 0], ['Dataset', 'sklearn k-means',
                                 joint_kind.upper() + ' + MF / input',
                                 joint_kind.upper() + ' + MF / latent']):
    ax.set_ylabel(label)
fig.tight_layout()
plt.show()

save_figure = False
if save_figure:
    Path('results').mkdir(exist_ok=True)
    fig.savefig('results/synthetic_comparison.png', dpi=180, bbox_inches='tight')


## Fixed-feature MF comparison

This keeps the earlier experiment in which the VAE was pretrained and frozen before fitting the factors. Set `run_fixed_mf` to compare it with joint training.


In [ ]:
run_fixed_mf = False
if run_fixed_mf:
    fixed_scores = []
    for name, (_, y) in zip(datasets_name, datasets_list):
        Z = representations[(name, 'vae')]['Z']
        result = train_mf_binary_model(Z, n_clusters=2, epochs=200, seed=seed)
        if name != 'no_structure':
            fixed_scores.append({'dataset': name, **score_dict(y, result['pred'])})
    display(pd.DataFrame(fixed_scores))


## Binary penalty illustrations

These illustrate the shapes discussed in the thesis. The adaptive threshold depends on the largest two entries of a row. It encourages separation but does not guarantee exactly one nonzero entry.


In [ ]:
x = np.linspace(-.25, 1.25, 600)
inside = (x >= 0) & (x <= 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(x, x**2 * (1 - x)**2, label='Mexican hat (lambda=2)')
axes[0].plot(x[inside], 1 - np.abs(1 - 2*x[inside]), label='PRIMP on [0, 1]')
axes[0].plot(x, np.minimum(np.abs(x) + .25*x*x,
                           np.abs(x - 1) + .25*(x - 1)**2), label='ELB')
axes[0].set(xlabel='y', ylabel='Penalty')
axes[0].legend()
row = torch.tensor([[.15, .35, .7, .9]])
root = row.topk(2, dim=1).values.mean().item()
axes[1].plot(x, x - root, label='row-adaptive direction')
axes[1].axvline(root, color='gray', ls='--', label=f'threshold = {root:.2f}')
axes[1].axhline(0, color='gray', lw=.6)
axes[1].set(xlabel='y', ylabel='Update direction')
axes[1].legend()
fig.tight_layout()
plt.show()
